In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from pandas import ExcelWriter
from bs4 import BeautifulSoup
import requests
import datetime
from selenium import webdriver
from time import sleep
import os




In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'EC SIBE' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
## to comment for the production environment
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running EC SIBE Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [7]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')



url = 'https://www.superbancos.gob.ec/bancos/catastro-publico/'




Typology ={

    	regulatorName +' 1': 'Bancos Públicos', 
		regulatorName +' 2': 'Bancos Privados Extranjeros', 
		regulatorName +' 3': 'Bancos Privados Nacionales', 
            
}


In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict



In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------



headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36 Edg/141.0.0.0'
}
response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')


else:
    print(f"[ERROR] Failed to fetch with status code {response.status_code}")
            # Find all tab titles
tabs = soup.find_all('div', class_='elementor-tab-title' ,attrs={'role': 'tab'})
#tabs = soup.find_all('div', attrs={'role': 'tab'})
for tab in tabs[:3]:
    aria_id = tab.get('aria-controls')
    tab_title = tab.text.strip()
    print(f"Tab: {tab_title} -> ID: {aria_id}")

    # Find the corresponding content div using the aria-controls ID
    content_div = soup.find('div', id=aria_id)
    if content_div:
        print(f"[FOUND] Content for '{tab_title}'")  # Preview first 500 chars
    else:
        print(f"[WARNING] No content found for '{tab_title}' with ID '{aria_id}'")
    
    key = next((k for k, v in Typology.items() if v == tab_title), '')

    print(key)  # Output: regulatorName + ' 3'

    table = content_div.find('tbody')
    tds = table.find_all('td')
    for td in tds:
        name = td.text
        if len(name.strip())<2:
            continue
        sqldict['Name'].append(name)
        sqldict['ListProcessDate'].append(processdate)   
        sqldict['RegulationType'].append('Regulated')
        sqldict['RegCtry'].append(key.split()[0])
        sqldict['RegCode'].append(key.split()[1])
        sqldict['ListCode'].append(key.split()[2])
        sqldict['ListName'].append(Typology[key])
sqldict = bourange_same_length_array(sqldict)


Tab: Bancos Privados Nacionales -> ID: elementor-tab-content-9371
[FOUND] Content for 'Bancos Privados Nacionales'
EC SIBE 3
Tab: Bancos Privados Extranjeros -> ID: elementor-tab-content-9372
[FOUND] Content for 'Bancos Privados Extranjeros'
EC SIBE 2
Tab: Bancos Públicos -> ID: elementor-tab-content-9373
[FOUND] Content for 'Bancos Públicos'
EC SIBE 1


In [9]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(writer, 'SQL Ready', index=False)
writer.save()
writer.close()
driver.quit()
sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_30876\802773129.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [11]:
df.to_csv('total_data.csv')

In [10]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 28 values.
Key 'priority' has 28 values.
Key 'ListLabel' has 28 values.
Key 'Typology' has 28 values.
Key 'EntryType' has 28 values.
Key 'Name' has 28 values.
Key 'InternalID_1' has 28 values.
Key 'InternalID_1_type' has 28 values.
Key 'InternalID_2' has 28 values.
Key 'InternalID_2_type' has 28 values.
Key 'InternalID_3' has 28 values.
Key 'InternalID_3_type' has 28 values.
Key 'CoType' has 28 values.
Key 'License_Type' has 28 values.
Key 'Address_1' has 28 values.
Key 'Address_2' has 28 values.
Key 'City' has 28 values.
Key 'Zip' has 28 values.
Key 'Cntry' has 28 values.
Key 'Phone' has 28 values.
Key 'Fax' has 28 values.
Key 'Website' has 28 values.
Key 'Email' has 28 values.
Key 'RegulationType' has 28 values.
Key 'RegulationTypeCode' has 28 values.
Key 'RegulationDate' has 28 values.
Key 'CancellationDate' has 28 values.
Key 'RegCtry' has 28 values.
Key 'RegCode' has 28 values.
Key 'ListCode' has 28 values.
Key 'ListLanguage' has 28 values.
Key 'ListValidityDate' h

In [ ]:
df.to_csv('ES_SIBE_v1.csv')